# Loss Functions Note: 1    
#### Abir Hossain | May, 2026

- This notebook has notes on the different activation functions and definition-details about them. The purpose of this notebook is to create a solid introduction to the different methods and usecases along with the reasoning behind. 
- The function types:
  + Regression Loss Functions
    * MSE
    * MAE
    * Huber Loss
    * Log-Cosh Loss
    * Quantile Loss
  + Classification Loss Functions
    * BCE
    * CCE
    * Hinge Loss
    * Focal Loss
  + Metric Learning & Representation Learning
    * Contrastive Loss
    * Triplet Loss
    * InfoNCE / NT-Xent
  + Sequence Modeling
    * CTC Loss
  + Generative Models
    * Adversarial Loss
    * KL Divergence Loss
    * Perceptual Loss
  + Computer Vision (Segmentation)
    * Dice/IoU Loss
  + More

# Mean Squared Error (MSE) / L2 Loss
- MSE measures the average of the squared differences between predicted and actual values. Squaring ensures:
   + All errors are positive (no cancellation)
   + Larger errors are penalized disproportionately more than small errors
- **Equation:**   MSE = $\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
- **Pros:**
   + Differentiable everywhere; smooth gradient makes optimization easy
   + Has a unique global minimum (for linear regression)
   + Mathematically convenient; analytically tractable
   + Heavily penalizes large errors, which is useful when outliers are genuinely important
- **Disadvantages:**
   + Extremely sensitive to outliers
   + Not robust in real-world noisy data
- **Usecases:**
   + When outliers represent genuinely important cases that must not be ignored
   + Baseline regression problems with relatively clean data
   + When a smooth, well-behaved loss surface is needed for optimization

# Mean Absolute Error (MAE) / L1 Loss
MAE measures the average of the absolute differences.
- **Equation:**    MAE = $\frac{1}{n}\sum_{i=n}^n |y_i - \hat{y}_i|$
- **Note:** The gradient is undefined at $y_i = \hat{y}_i$. In practice, we define it as 0 or a small value at that point.
- **Pros:**
  + Robust to outliers; large errors are not exaggerated
  + Error is in the same units as the target variable
  + Minimizing MAE yields the conditional median of the target
- **Disadvantages:**
  + Non-differentiable at zero; can cause optimization instability
  + Gradient is constant regardless of error magnitude, so the model receives the same "signal" for a small error as for a large error
  + Can be harder to converge for gradient-based methods
- **Usecases:**
  + When outliers are noise that should be ignored
  + When interpretability of error in original units matters
  + Financial forecasting, weather prediction, or any domain with noisy extreme values

# Huber Loss
Huber loss is a hybrid of MSE and MAE:
 + For small errors ($\le \delta$): behaves like MSE (quadratic, smooth, easy to optimize)
 + For large errors ($> \delta$): behaves like MAE (linear, robust to outliers)
- **Equation:**
$$
L_\delta =
\begin{cases}
    \frac{1}{2}(y - \hat{y})^2  \; & \mbox{for} \; |y - \hat{y}| \le \delta\\
    \delta(|y - \hat{y}| - \frac{1}{2}\delta) \; & \mbox{for} \; |y - \hat{y}| > \delta
\end{cases}
$$
- **Pros:**
  + Smooth near the optimum (like MSE) but robust to outliers (like MAE)
  + Differentiable everywhere (unlike MAE)
  + The $\delta$ parameter gives control over the robustness threshold
- **Cons:**
  + Requires tuning the $\delta$ hyperparameter
  + Slightly more computationally expensive
- **Usecases:**
  + When outliers exist but still smooth optimization is needed
  + Robotics, sensor data, autonomous driving; domains with noisy measurements

# Log-Cosh Loss
Log-cosh is approximately $\frac{1}{2}(y - \hat{y})^2$ for small errors and approximately $∣y−\hat{y}∣ −\: log(2)$  for large errors. It's like a smoother version of Huber loss that doesn't require a hyperparameter.
- **Equation:** $\; \;$ ***LogCosh*** = $\sum_{i=1}^{n}log(cosh(\hat{y_i} - y_i)) \; $
| $\;$ Where, cosh(x) = $\frac{e^x + e^{-x}}{2}$ 
- **Pros:**
  + Twice differentiable everywhere (even smoother than Huber)
  + Less sensitive to outliers than MSE
  + No hyperparameters to tune
- **Cons:**
  + Computationally more expensive as it involves exponential functions
- **Usecases:**
  + Robust regression without tuning hyperparameters
  + As a drop-in replacement for MSE in the case of moderate outliers

# Quantile Loss / Pinball Loss
Quantile loss is asymmetric. It penalizes under-prediction and over-prediction differently depending on the chosen quantile **q** :
|Scenario|Penalty|
|---|---|
|Under-prediction ($y > \hat{y}$)|Weighted by ***q***|
|Over-prediction ($y < \hat{y}$)|Weighted by ***(1−q)***| 
Meaning:
  + If q=0.5  (median): Under and over predictions are penalized equally;equivalent to MAE (scaled by 0.5)
  + If q=0.9 : Under-predictions are penalized 9× more heavily than over-predictions. The model will tend to over-predict to avoid costly under-predictions.
  + If q=0.1 : Over-predictions are penalized 9× more heavily than under-predictions. 
- **Equation:** $\; L_q = \frac{1}{n} \sum_{i=1}^{n} \begin{cases} 
                            q \cdot (y_i - \hat{y}_i) \; & \mbox {if} \;  y_i \ge \hat{y}_i \\
                            (1 - q) \cdot (\hat{y}_i - y_i) \; & \mbox {if} \;  y_i < \hat{y}_i
\end{cases}$
- **Pros:**
  + Gives full predictive distributions, not just point estimates. This captures uncertainty naturally.
  + Real-world decisions often have asymmetric costs; which quantile loss lets encode directly into the optimization.
  + Unlike MSE (which implicitly assumes Gaussian errors), quantile regression makes no assumption about the error distribution. Works for skewed, heteroscedastic, or heavy-tailed data.
  + Extreme quantiles focus on specific tails and ignore the opposite tail.
  + A prediction at quantile q=0.9  means "we are 90% confident the true value will be below this prediction".
- **Cons:**
  + Like MAE, the gradient is piecewise constant, which can make optimization harder with gradient descent. Requires specialized solvers or smoothing techniques.
  + To get a complete picture (e.g., 10th, 50th, 90th percentiles), training separate models for each quantile is common. Some modern approaches try to address this.
  + When training separate models, might get logically impossible results. Requires post-processing or constrained optimization to fix
  + If errors are truly Gaussian and symmetric, MSE is the maximum likelihood estimator and statistically more efficient
- **Usecases:**
  + Demand forecasting	
  + Financial risk modeling	
  + Weather prediction	
  + Any domain with asymmetric costs	
  + Heteroscedastic data	


# Binary Cross-Entropy (Log Loss)
Cross-entropy measures the difference between two probability distributions. It heavily penalizes confident wrong predictions. 
- **Equation:**
$\; BCE = - \frac{1}{n}\sum_{i=1}^{n}[y_i\cdot log(\hat{y}_i) + (1 - y_i)\cdot log(1 - \hat{y}_i)]$   
Where, $y_i \in$ {0, 1} and $\hat{y}_i\in {0, 1}$
- **Pros:**
  + Strong gradients for very wrong predictions
  + Minimizing BCE is equivalent to **Maximum Likelihood Estimation**
  + Natural fit for binary classification with sigmoid activation
- **Cons:**
  + Requires predicted probabilities; Sigmoid Activation
  + Can suffer from numerical instability with probabilities very close to 0 or 1
  + Assumes all misclassifications are equally bad
- **Usecases:**
  + Binary classification
  + When probabilistic outputs are needed
  + When confident wrong answers should be penalized severely

# Categorical Cross-Entropy
Same as binary cross-entropy but extended to multiple classes. Only the log probability of the correct class contributes to the loss.
- **Equation:**
$CCE = - \frac{1}{n}\cdot \sum_{i=1}^{n} \sum_{c=1}^{C}y_{i,c}log(\hat{y}_{i,c})$  
Where:  
    C  = number of classes  
    $y_{i,c}$  = 1 if sample i  belongs to class c , else 0 (one-hot)  
    $\hat{y}_{i,c}$  = predicted probability for class c   
- **Usecases:**
  + Multi-class classification
  + With softmax activation in the output layer
- **Variants:**
  +  Sparse Categorical Cross-Entropy: more memory efficient
  +  Label Smoothing: prevent overconfidence

# Hinge Loss
The loss is zero if the prediction is correct and confident. If the prediction is wrong or not confident enough, the loss increases linearly.
- **Equation:**
$Hinge = \frac{1}{n}\sum_{i=1}{n} max(0, 1 - y_i\cdot \hat{y}_i)$ 
Where, $y_i \in {-1, +1}$  
- **Pros:**
   + Focuses on maximizing the margin between classes
   + Ignores correctly classified points that are far from the decision boundary
   + Robust to outliers on the correct side of the margin
- **Cons:**
   + Only meaningful for binary classification, extensions exist but are complex
   + Not probabilistic 
   + Non-differentiable at the hinge point, though subgradient methods work
- **Usecases:**
   + SVM training
   + When the margin/decision boundary is important more than probabilities
   + Binary classification with linear models

# Focal Loss
An extension of cross-entropy designed for imbalanced datasets. 
- **Equation:**
FL = $-\frac{1}{n}\sum_{i=1}^{n}\alpha(1 - \hat{y}_i)^\gamma \cdot y_i log(\hat{y}_i)$ 
  + Where:  
    $\gamma \ge 0$  = focusing parameter, typically 2  
    $\alpha$  = weighting factor for class imbalance  
  + The $(1 - \hat{y}_i)^\gamma$ term down-weights easy examples, where the model is already confident, so that training focuses on hard, misclassified examples.
    * When $\gamma$ = 0 ; Focal Loss = Cross-Entropy
    * As $\gamma$ increases; easy examples are down-weighted more aggressively
- **Usecases:**
  + Object detection (e.g., RetinaNet)
  + Highly imbalanced classification (fraud detection, rare disease diagnosis)
  + When easy negative examples dominate the loss

# Contrastive Loss (Chopra et al., 2005)

- **Equation:**
$\mathcal{L}$ = $\frac{1}{N}\sum_{i=1}^{N}[y_i\cdot d^2_{ij} + (1 - y_i)\max(0, m - d_{ij})^2]$   
Where:  
    $y_i$ = 1  if the pair *(i,j)*  is similar *(positive)* and 0  if dissimilar *(negative)*  
    $d_{ij}$ = $∣∣f(x_i) − f(x_j)∣∣_2$ = Euclidean distance between embeddings  
    m  = margin (hyperparameter; Like 1.0 or 2.0)  
    Positive pairs: Minimize distance, pull together  
    Negative pairs: Push apart, but only if they are already closer than margin m . If they are already far apart (distance $>$ m ), the loss is zero.   
- **Pros:**
   + Simple, intuitive geometric interpretation
   + Margin prevents the model from trivially pushing all embeddings to infinity
- **Cons:**
   + Requires carefully curated pairs; random negatives are often too easy and provide no gradient
   + Sampling strategy is everything; hard negative mining is essential
   + Doesn't naturally extend to more than two samples at a time
- **Usecases:**
   + Siamese networks for signature verification, face matching, duplicate detection
   + Early metric learning approaches which is mostly superseded by **triplet** and **InfoNCE** now

# Triplet Loss (Schroff et al., FaceNet 2015)
- **Equation:**  
$\mathcal{L}$ = $\sum_{i}^{N} [||f(x_i^a) - f(x_i^p)||_2^2 - ||f(x_i^a) - f(x_i^n)||_2^2 + \alpha]_+$  
Where:  
    $x^a$  = anchor ; reference sample  
    $x^p$  = positive ; same class as anchor  
    $x^n$  = negative ; different class from anchor  
    $\alpha$  = margin  
    $[z]_+$ = max(0,z)  = hinge function  
    Instead of comparing raw pairs, triplet loss compares relative distances. The goal is: $||anchor - positive||^2 + \alpha < ||anchor - negative||^2$  
    The anchor must be closer to the positive than to the negative by at least margin $\alpha$.
- **Pros:**
    + Learns relative relationships, not just absolute similarity
    + Embeddings naturally cluster by *identity/class* in the space
    + Very effective for face recognition and person re-identification
- **Cons:**
    + Extremely sensitive to triplet sampling. Most random triplets satisfy the margin easily and yield zero loss. Hard negative mining *(select negatives that are currently close to the anchor)* or semi-hard negative mining *(negatives that violate the margin but are not the closest possible)* is needed.
    + Training can be unstable if triplets are too hard
    + Computationally expensive to find good triplets
- **Usecases:**
    + Face recognition/verification (FaceNet, DeepID)
    + Person re-identification
    + Any task where object/person verification is necessary rather than classification

# InfoNCE / NT-Xent (Normalized Temperature-scaled Cross Entropy)
- [Read a document about it](https://medium.com/self-supervised-learning/nt-xent-loss-normalized-temperature-scaled-cross-entropy-loss-ea5a1ede7c40)

- **Pros:**
  + No need for explicit negative mining; the entire batch serves as negatives
  + Scales beautifully with batch size
  + Theoretical connection to mutual information maximization
  + State-of-the-art for self-supervised learning
- **Cons:**
  +  Requires very large batch sizes (thousands) to get enough negatives
  +  Performance heavily depends on data augmentation quality (SimCLR, MoCo)
  +  Can suffer from false negatives; though this is usually minor
- **Usecases:**
  +  Self-supervised visual representation learning (SimCLR, MoCo, CLIP)
  +  Contrastive pre-training for NLP (sentence embeddings, ELECTRA-style)
  +  Any scenario where pairs of related samples are there and learning representations without labels is needed

# CTC Loss (Connectionist Temporal Classification)
In speech recognition or handwriting, the input sequence (audio frames, pixels) is much longer than the output sequence (characters, phonemes). We don't know which frame maps to which character.
- CTC solves this by:
   + Allowing the model to output blank tokens
   + Allowing repeated characters to be collapsed
   + Summing probabilities over all valid alignments automatically
   + Example:
     * Target: "cat"
     * Possible alignments: cc--aaa-ttt, c-a-a-t---, -c--a-t--t, etc.
     * CTC computes the total probability of all paths that collapse to "cat"
- **Equation:**
$\; \mathcal{L}_{CTC} = - \sum_{x,z\in \mathcal{D}} log P(z|x)$  
Where ***P(z∣x)***  is computed by summing over all possible alignments of the target sequence z  with the input sequence x  using a dynamic programming algorithm *(Forward-Backward)*.
- **Pros:**
   + No need for frame-level alignment labels; only the final transcript is needed
   + End-to-end differentiable
   + Works for ***variable-length input : variable-length output***
- **Cons:**
   + Assumes conditional independence between outputs at different time steps; given the input. The model cannot learn that "q" is almost always followed by "u" directly in the CTC layer; though the encoder can learn this.
   + Tends to produce ***peaky*** outputs that are overconfident at single frames
   + Struggles with language modeling; purely acoustic; often needs an external language model for best results
- **Usecases:**
   + Speech recognition (DeepSpeech, wav2letter)
   + Handwriting recognition
   + OCR on unsegmented text
   + Any sequence-to-sequence problem where alignment is unknown and monotonic
- **Modern Alternatives:**
   + Attention-based seq2seq (Listen, Attend, Spell) where ***CTC*** is replaced with ***attention alignment***
   + RNN-T *(Transducer)*; combines ***CTC*** with a prediction network for better language modeling
   + CTC + Attention hybrid (ESPnet)

# Adversarial Loss (GAN Loss)
- [Read a document about it](https://medium.com/analytics-vidhya/understanding-gans-deriving-the-adversarial-loss-from-scratch-ccd8b683d7e2)
- **Pros:**
   + Can model incredibly complex, high-dimensional distributions
   + Produces sharp, realistic samples (images, audio)
   + No explicit density modeling needed
- **Cons:**
   + Notoriously unstable to train: ***mode collapse, vanishing gradients, non-convergence***
   + No explicit likelihood; hard to evaluate quantitatively
   + Requires careful balancing of **G** and **D** training steps
- **Usecases:**
   + Image generation (StyleGAN, BigGAN)
   + Image-to-image translation (pix2pix, CycleGAN)
   + Super-resolution, inpainting, style transfer
   + Any generative task where sample quality matters more than exact likelihood
- **Modern Variants:**
   + WGAN (Wasserstein GAN): Uses Wasserstein distance instead of JS divergence; much more stable
   + WGAN-GP: Gradient penalty instead of weight clipping
   + LSGAN (Least Squares GAN): Uses MSE instead of log loss; more stable
   + RaGAN (Relativistic GAN): Discriminator compares real vs fake relatively

# KL Divergence (Kullback-Leibler Divergence)
- [Read a document about it](https://www.datacamp.com/tutorial/kl-divergence)
- **Pros:**
   + Well-founded in information theory
   + Provides a principled way to regularize distributions
   + Differentiable and easy to compute for common distributions; Gaussian KL has a closed form
- **Cons:**
   + Asymmetric; direction matters. In VAEs,$D_{KL}$(q∣∣p) is used *(mean-seeking, zero-avoiding)*, which tends to cover the prior but may be too broad.
   + Undefined / infinite; if Q(x)=0  where P(x)$>$0 
   + Can lead to posterior collapse in VAEs; especially in language modeling
- **Usecases:**
   + VAE training; as regularization
   + Probabilistic model training
   + Distribution matching tasks
   + As a component; rarely as the sole loss

# Perceptual Loss (Feature Matching Loss)
Instead of comparing pixels directly (MSE), compare high-level features extracted by a pre-trained CNN. The idea aligns with human perception: *we care about semantic content and style, not exact pixel values*. So two images are ***similar*** if they produce similar neural activations, even if pixels differ.
- **Equation:**
$\mathcal{L}_{perceptual} = \sum_l ||\phi_l(\hat{x}) - \phi_l(x)||_2^2$   
Where:   
    x  = ground truth image   
    $\hat{x}$  = generated/reconstructed image  
    $\phi_l$  = activations from layer ***l***  of a pre-trained network; usually VGG16/19 ImageNet features  
    The sum is over multiple layers; typically early layers for texture, deeper layers for structure/semantics  
- **Pros:**
    + Produces visually sharper results than MSE/MAE; which lead to blurry outputs due to averaging multiple plausible solutions
    + Captures texture, style, and semantic content
    + No additional trainable parameters
- **Cons:**
    + Depends on the pre-trained network being relevant to the domain
    + Computationally more expensive than pixel losses
    + Can introduce artifacts from the pre-trained network's biases
    + Not a proper mathematical metric
- **Usecases:**
    + Image super-resolution (SRGAN, ESRGAN)
    + Neural style transfer
    + Image inpainting
    + Any ***image-to-image*** translation where perceptual quality matters more than pixel-perfect accuracy
- **Related:** Gram Matrix Loss (Style Loss)  
$L_{style} = \sum_l∣∣G_l(\hat{x}) − G_l(x)∣∣_F^2 $   
Where $G_l$  is the Gram matrix of features at layer l , capturing texture/style correlations. Used alongside perceptual loss in neural style transfer.

# Dice Loss / IoU Loss
- **Equations:**
    + $\mathcal{L}_{Dice} = 1 - \frac{2 \sum_{i=1}^{N} p_i \cdot g_i}{\sum_{i=1}^{N} p_i^2 + \sum_{i=1}^{N} g_i^2 }$   
    + $\mathcal{L}_{IoU} = 1 - \frac{\sum p_i \cdot g_i}{\sum p_i + \sum g_i - \sum p_i \cdot g_i}$   
        Where:  
        $p_i \in [0, 1]$   = predicted probability for pixel *i*   
        $g_i \in {0,1}$  = ground truth mask for pixel *i*   
        Numerator = intersection  
        Denominator = sum of predictions + sum of ground truths  
    Both directly optimize the overlap between predicted and ground-truth segmentation masks rather than per-pixel accuracy.  
    Source:  
    ***Dice coefficient (F1-score for segmentation)*** = $\frac {2|X \cap Y|}{|X| + |Y|}$    
    ***IoU (Jaccard index)*** = $\frac {|X \cup Y|}{X \cap Y}$     
- **Pros:**
    + Handles extreme class imbalance naturally. *eg. in medical imaging, a tumor might be 1% of the image. Cross-entropy would be 99% satisfied by predicting all background. Dice loss forces the model to care about the small foreground region.*
    + Directly optimizes the important evaluation metric 
    + Smooth and differentiable
- **Cons:**
    + Can be unstable when both prediction and ground truth are very small as denominator approaches zero
    + Less stable gradients than cross-entropy in early training
    + Often combined with cross-entropy for best results
- **Usecases:**
    + Medical image segmentation (tumors, organs)
    + Satellite imagery (small objects in large scenes)
    + Any semantic/instance segmentation with severe class imbalance

# ArcFace / CosFace / SphereFace (Additive Angular Margin Loss)
Classification loss for face recognition that adds a margin in angular space.
- Enforces intra-class compactness and inter-class separability in the angular space. State-of-the-art for face recognition; ***ArcFace*** is the current standard - more than ***Triplet Loss***. 

# Supervised Contrastive Loss (SupCon)
Combines labels with contrastive learning. Instead of treating only the augmented view as positive, it treats all samples from the same class as positives.
- Outperforms cross-entropy on ImageNet classification while learning much better representations. The paper - *Khosla et al., 2020* - showed that contrastive learning benefits enormously from labels.

# CLIP Contrastive Loss
Cross-modal InfoNCE between images and text. Given a batch of N  (image, text) pairs, learn to align them in a joint embedding space.
- Powers DALL-E, Stable Diffusion, and virtually all modern vision-language models and multimodal AI.

# Diffusion Loss (Noise Prediction Loss)
In diffusion models (DDPM, Stable Diffusion), the loss is simply MSE between predicted noise and actual noise.
- This deceptively simple loss underpins the entire generative AI revolution (DALL-E 2/3, Midjourney, Stable Diffusion).
- Modern generative models.

# Consistency Loss (Consistency Models)
Distillation loss that enforces self-consistency.
- Allows single-step generation (unlike diffusion which needs 50+ steps). Emerging as the next generation after diffusion.

# Self-Supervised Losses
|Loss|Core Idea|
|---|---|
|Barlow Twins|***Redundancy reduction:*** make cross-correlation matrix of two augmented views close to identity|
|VicReg|***Variance-Invariance-Covariance:*** ensures embeddings vary across samples, are invariant to augmentation, and have decorrelated dimensions|
|BYOL|***Bootstrap Your Own Latent:*** online network predicts target network output; no negatives needed|
- These are the modern successors to SimCLR/InfoNCE that do not require negative samples. Critical for self-supervised learning research.

# Sinkhorn Divergence / Optimal Transport Losses
Differentiable approximation of Wasserstein distance using entropy-regularized optimal transport.
- More stable than GANs for generative modeling. Used in Wasserstein Autoencoders (WAE) and score-based generative models.

# Ranking Losses (ListNet, LambdaRank, RankNet)
Losses for learning-to-rank. Not classification or regression — directly optimize ranking quality (NDCG, MAP).
- Essential for search, recommendation, and information retrieval.
- Recommender systems or search engines.